In [1]:
"""
- ID3 (Iterative Dichotomiser 3) was developed in 1986 by Ross Quinlan. The algorithm creates a multiway tree, finding 
      for each node (i.e. in a greedy manner) the categorical feature that will yield the largest information gain for 
      categorical targets. Trees are grown to their maximum size and then a pruning step is usually applied to improve 
      the ability of the tree to generalise to unseen data.

- C4.5 is the successor to ID3 and removed the restriction that features must be categorical by dynamically defining a 
      discrete attribute (based on numerical variables) that partitions the continuous attribute value into a discrete 
      set of intervals. C4.5 converts the trained trees (i.e. the output of the ID3 algorithm) into sets of if-then rules.
      These accuracy of each rule is then evaluated to determine the order in which they should be applied. Pruning is 
      done by removing a rule’s precondition if the accuracy of the rule improves without it.

- C5.0 is Quinlan’s latest version release under a proprietary license. It uses less memory and builds smaller rulesets 
      than C4.5 while being more accurate.

- CART (Classification and Regression Trees) is very similar to C4.5, but it differs in that it supports numerical 
      target variables (regression) and does not compute rule sets. CART constructs binary trees using the feature and 
      threshold that yield the largest information gain at each node.

*** scikit-learn uses an optimised version of the CART algorithm ***
Documentation: https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html
"""

%matplotlib inline
import warnings
from sklearn.exceptions import UndefinedMetricWarning
warnings.filterwarnings("error", category=UndefinedMetricWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)
import matplotlib.pyplot as plt
import pandas as pd
from sklearn import tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import make_scorer
from sklearn.metrics import precision_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import recall_score
from sklearn.metrics import fbeta_score
from sklearn.metrics import auc
from sklearn.model_selection import StratifiedKFold
from sklearn import metrics
from sklearn.tree import export_graphviz
from sklearn.externals.six import StringIO  
from IPython.display import Image  
import pydotplus
import pickle

def PlotTree(clf, features, name="Tree.png"):
    dot_data=StringIO()
    export_graphviz(clf, out_file=dot_data, filled=True, rounded=True, special_characters=True,
                    feature_names=features,class_names=['0.0','1.0'])
    graph = pydotplus.graph_from_dot_data(dot_data.getvalue())  
    graph.write_png(name)
    return Image(graph.create_png())

def GetMetrics(y_test, y_pred):
    print("Accuracy:",metrics.accuracy_score(y_test, y_pred))
    display("confusion_matrix", pd.DataFrame(metrics.confusion_matrix(y_test, y_pred)))
    report = metrics.classification_report(y_test, y_pred, output_dict=True)
    display("metrics", pd.DataFrame(report).transpose())


In [2]:
## Load Dataset ....

Class=["refatoracao"]
features = [ "LOC", "NPM", "WMC", "NOC", "DIT", "CAM", 
             "CBO", "RFC", "CA", "CE", "LCOM3", #"LCOM",
             "DAM", "MOA", "MFA", "IC", "CBM", "AMC", 
           ]
df = pd.read_csv('dataset_refat_final.csv',delimiter=";", decimal=".") #, header=True) #,names=features)
x = df[features]
y = df[Class]
print("Data Size: ",df.shape)
df[features+Class].head()

Data Size:  (2030, 25)


,LOC,NPM,WMC,NOC,DIT,CAM,refatoracao
0,1656.0,88.0,100.0,0.0,0.0,0.2148,1.0
1,253.0,11.0,14.0,4.0,1.0,0.1688,1.0
2,1421.0,88.0,96.0,0.0,0.0,0.2211,1.0
3,517.0,24.0,30.0,0.0,0.0,0.2867,1.0
4,783.0,17.0,22.0,0.0,0.0,0.2273,1.0


In [3]:
# Default params for DecisionTreeClassifier and split randomly the dataset in training and test (70% training, 30% test)

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=1) 
clf = DecisionTreeClassifier(random_state=0)
clf = clf.fit(X_train,y_train)
y_pred = clf.predict(X_test)

GetMetrics(y_test, y_pred)
img=PlotTree(clf, features, "Tree.png")

Accuracy: 0.8078817733990148


'confusion_matrix'

,0,1
0,257,51
1,66,235


'metrics'

,f1-score,precision,recall,support
0.0,0.814580,0.795666,0.834416,308.0
1.0,0.800681,0.821678,0.780731,301.0
micro avg,0.807882,0.807882,0.807882,609.0
macro avg,0.807631,0.808672,0.807573,609.0
weighted avg,0.807711,0.808522,0.807882,609.0


In [4]:
## Find the best params for DecisionTreeClassifier and we use KFold

dec_tree = DecisionTreeClassifier(random_state=0) # random_state - deterministic behaviour during fitting random_state has to be fixed to an integer
pipe = Pipeline(steps=[ ('dec_tree', dec_tree),
                      ])
parameters = dict(dec_tree__criterion=['gini', 'entropy'],
                  dec_tree__min_samples_split=[2, 4, 8, 16, 32],
                  dec_tree__min_samples_leaf=[2, 4, 8, 16, 32],
                  dec_tree__max_depth=[2, 4, 6, 8, 10, 12, 14, 16, 18],
                  dec_tree__max_features=list(range(1,len(features)+1)),
                 )
scores = {'accuracy' :make_scorer(accuracy_score),
          'recall'   :make_scorer(recall_score),
          'precision':make_scorer(precision_score),
          'f1'       :make_scorer(fbeta_score, beta = 1),
          'auc'      :make_scorer(auc),
         }
KF = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)
clf_GS = GridSearchCV(pipe, param_grid = parameters, n_jobs=-1, cv=KF,
                      #scoring = scores,   refit = 'auc', 
                      scoring = 'roc_auc',
                     )
clf_GS_fit=clf_GS.fit(x, y)
pickle.dump(clf_GS, open("BestDecisionTree", 'wb')) # Save Model

print("Decision tree tuning")
print("Steps:\n",pipe.steps)
print("Best Parameters:\n\t Params: {}\n\t Score: {}\n".format(clf_GS.best_params_,clf_GS.best_score_))

print("** metrics for ALL data set")
y_pred = clf_GS_fit.predict(x)
GetMetrics(y, y_pred)

print("** metrics for the test data set (previous model - 70% training and 30% test)")
y_pred = clf_GS_fit.predict(X_test)
GetMetrics(y_test, y_pred)

img=PlotTree(clf_GS.best_estimator_.named_steps["dec_tree"], features, "Tree_GS.png")


Decision tree tuning
Steps:
 [('dec_tree', DecisionTreeClassifier(class_weight=None, criterion='gini', max_depth=None,
            max_features=None, max_leaf_nodes=None,
            min_impurity_decrease=0.0, min_impurity_split=None,
            min_samples_leaf=1, min_samples_split=2,
            min_weight_fraction_leaf=0.0, presort=False, random_state=0,
            splitter='best'))]
Best Parameters:
	 Params: {'dec_tree__criterion': 'gini', 'dec_tree__max_depth': 6, 'dec_tree__max_features': 3, 'dec_tree__min_samples_leaf': 32, 'dec_tree__min_samples_split': 2}
	 Score: 0.9178770351650656

** metrics for ALL data set
Accuracy: 0.8729064039408867


'confusion_matrix'

,0,1
0,836,172
1,86,936


'metrics'

,f1-score,precision,recall,support
0.0,0.866321,0.906725,0.829365,1008.0
1.0,0.878873,0.844765,0.915851,1022.0
micro avg,0.872906,0.872906,0.872906,2030.0
macro avg,0.872597,0.875745,0.872608,2030.0
weighted avg,0.872641,0.875531,0.872906,2030.0


** metrics for the test data set (previous model - 70% training and 30% test)
Accuracy: 0.8620689655172413


'confusion_matrix'

,0,1
0,257,51
1,33,268


'metrics'

,f1-score,precision,recall,support
0.0,0.859532,0.886207,0.834416,308.0
1.0,0.864516,0.840125,0.890365,301.0
micro avg,0.862069,0.862069,0.862069,609.0
macro avg,0.862024,0.863166,0.862391,609.0
weighted avg,0.861995,0.863431,0.862069,609.0


In [5]:
TrainParmResults=pd.DataFrame(clf_GS.cv_results_)
pd.options.display.max_columns = None
TrainParmResults.head(6) # -1 Show all 

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_dec_tree__criterion,param_dec_tree__max_depth,param_dec_tree__max_features,param_dec_tree__min_samples_leaf,param_dec_tree__min_samples_split,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,split5_test_score,split6_test_score,split7_test_score,split8_test_score,split9_test_score,mean_test_score,std_test_score,rank_test_score,split0_train_score,split1_train_score,split2_train_score,split3_train_score,split4_train_score,split5_train_score,split6_train_score,split7_train_score,split8_train_score,split9_train_score,mean_train_score,std_train_score
0,0.009651,0.011337,0.002173,0.002340,gini,2,1,2,2,"{'dec_tree__criterion': 'gini', 'dec_tree__max...",0.861626,0.861723,0.834935,0.864735,0.843428,0.863279,0.873859,0.860318,0.855294,0.825686,0.854509,0.014234,2648,0.859178,0.865669,0.860272,0.866614,0.859217,0.866976,0.855786,0.857628,0.857948,0.869464,0.861875,0.00456
1,0.004861,0.003495,0.003241,0.003312,gini,2,1,2,4,"{'dec_tree__criterion': 'gini', 'dec_tree__max...",0.861626,0.861723,0.834935,0.864735,0.843428,0.863279,0.873859,0.860318,0.855294,0.825686,0.854509,0.014234,2648,0.859178,0.865669,0.860272,0.866614,0.859217,0.866976,0.855786,0.857628,0.857948,0.869464,0.861875,0.00456
2,0.004493,0.001719,0.002526,0.002053,gini,2,1,2,8,"{'dec_tree__criterion': 'gini', 'dec_tree__max...",0.861626,0.861723,0.834935,0.864735,0.843428,0.863279,0.873859,0.860318,0.855294,0.825686,0.854509,0.014234,2648,0.859178,0.865669,0.860272,0.866614,0.859217,0.866976,0.855786,0.857628,0.857948,0.869464,0.861875,0.00456
3,0.002724,0.001579,0.002873,0.001603,gini,2,1,2,16,"{'dec_tree__criterion': 'gini', 'dec_tree__max...",0.861626,0.861723,0.834935,0.864735,0.843428,0.863279,0.873859,0.860318,0.855294,0.825686,0.854509,0.014234,2648,0.859178,0.865669,0.860272,0.866614,0.859217,0.866976,0.855786,0.857628,0.857948,0.869464,0.861875,0.00456
4,0.001978,0.004349,0.003831,0.004662,gini,2,1,2,32,"{'dec_tree__criterion': 'gini', 'dec_tree__max...",0.861626,0.861723,0.834935,0.864735,0.843428,0.863279,0.873859,0.860318,0.855294,0.825686,0.854509,0.014234,2648,0.859178,0.865669,0.860272,0.866614,0.859217,0.866976,0.855786,0.857628,0.857948,0.869464,0.861875,0.00456
5,0.004323,0.001550,0.004583,0.001653,gini,2,1,4,2,"{'dec_tree__criterion': 'gini', 'dec_tree__max...",0.861626,0.861723,0.834935,0.864735,0.843428,0.863279,0.873859,0.860318,0.855294,0.825686,0.854509,0.014234,2648,0.859178,0.865669,0.860272,0.866614,0.859217,0.866976,0.855786,0.857628,0.857948,0.869464,0.861875,0.00456
